# 07 forge boot

Question: does `golem run` boot the Forge server of `tests/fetch.py` unchanged in its embedded JVM
with the bridge mod connected to the real natives of `golem-bridge`, answer the command `golem` of
the core plugin on the console and to a player, read the game, tell the sources of a command apart
and teleport and name a player through the tools of operators, and stop on a signal through the
server thread with the worlds saved, and does a server without the bridge mod still stop and save
on a signal.

The notebook builds `golem` with cargo and the bridge with Gradle, then boots copies of the
installed server twice under `golem run`, never under the java launcher. The first boot has the mod
in `mods/`: a player of `tests/harness/minecraft.py` joins, chats and runs `/golem source` and
`/golem status`, the console runs both too, the console, the player, an armor stand and a command
block run `/golem whoami`, the operators' subcommands read the game and teleport the player into
the Nether and name them in the player list, and `SIGTERM` stops the server while the player is
still online, with a `SIGINT` behind it that must change nothing. The second boot has no mod, so
`SIGTERM` must end the JVM through `System.exit` and the shutdown hook of the server, as under
java. `okf/jvm-host.md` describes the stop path and `okf/plugins/core.md` the command.

In [ ]:
# Papermill overrides this cell.
# The tree under test, the root of the repository.
golem_root = "."

# Gradle, a path relative to the root or a name on PATH. tests/fetch.py installs it here.
gradle = "tests/.cache/gradle-8.14.3/bin/gradle"
# The JDK 17 whose libjvm golem loads, which also runs Gradle.
java_home = "tests/.cache/jdk-17.0.21+9"
# The Forge server that tests/fetch.py installs. The notebook boots copies of it alone.
forge_server = "tests/.cache/forge-server"
# The port of the copies of the server, which bind 127.0.0.1 alone.
server_port = 25655
# True passes --offline to Gradle, for a machine whose Gradle cache is warm.
gradle_offline = False
# The seconds a boot may take to reach Done, and a stop to end.
boot_timeout = 300

# The pins.
forge_version = "47.4.18"
minecraft_version = "1.20.1"

## Setup

The cell finds the repository and the downloads, builds `golem` with cargo and reads what
`golem source` names: the version, the repository and the commit, which `build.rs` takes from
`git rev-parse HEAD` unless `GOLEM_COMMIT` is set.

In [ ]:
import json
import os
import re
import shutil
import signal
import subprocess
import sys
import tempfile
import threading
import time
import tomllib
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import QUIET, paths, run
from harness.minecraft import Client, offline_uuid

P = paths(golem_root)


def located(value):
    # A value with a slash is a path relative to the root, a bare name is looked up on PATH.
    if "/" in value:
        return (P.root / value).resolve()
    found = shutil.which(value)
    assert found, "%s is not on PATH" % value
    return Path(found)


GRADLE = located(gradle)
JAVA_HOME = located(java_home)
FORGE = located(forge_server)
for needed in (GRADLE, JAVA_HOME / "bin" / "java", FORGE):
    assert needed.exists(), "%s is missing, python3 -I tests/fetch.py installs it" % needed.name
SCRATCH = Path(tempfile.mkdtemp(prefix="golem-07-"))
ENVIRONMENT = dict(os.environ, **QUIET, JAVA_HOME=str(JAVA_HOME))
WORKSPACE = tomllib.loads((P.root / "Cargo.toml").read_text())["workspace"]["package"]


def shown(text):
    # The text with the temporary directory and the root replaced, so no machine path is printed.
    return text.replace(str(SCRATCH), "<scratch>").replace(str(P.root), "<root>")


run(["cargo", "build", "--quiet", "-p", "golem"], cwd=P.root, timeout=1800, check=True)
metadata = json.loads(run(["cargo", "metadata", "--format-version", "1", "--no-deps"], cwd=P.root, check=True).stdout)
GOLEM = Path(metadata["target_directory"]) / "debug" / "golem"
assert GOLEM.is_file(), "cargo built no golem"
source = dict(line.split(" ", 1) for line in run([GOLEM, "source"], check=True).stdout.splitlines())
COMMIT = source["commit"]
print("python:", sys.version.split()[0])
print("golem source:", source)
assert source["version"] == WORKSPACE["version"] and source["repository"] == WORKSPACE["repository"], source
assert len(COMMIT) == 40 or COMMIT == "unknown", COMMIT

## The bridge

The cell builds a copy of `bridge/` with Gradle, as `05_bridge_build` does and checks in depth,
and keeps `golem-host.jar` and the mod.

In [ ]:
TREE = SCRATCH / "tree"
shutil.copytree(P.root / "bridge", TREE / "bridge", ignore=shutil.ignore_patterns("build", ".gradle"))
shutil.copy2(P.root / "Cargo.toml", TREE / "Cargo.toml")
command = [GRADLE, "-p", TREE / "bridge", "--console=plain", "-Porg.gradle.java.installations.paths=%s" % JAVA_HOME,
           "build"] + (["--offline"] if gradle_offline else [])
started = time.monotonic()
build = subprocess.run([str(part) for part in command], capture_output=True, text=True, env=ENVIRONMENT, timeout=1700)
print("gradle build: exit %d after %.0f s" % (build.returncode, time.monotonic() - started))
assert build.returncode == 0, "the build failed:\n" + build.stdout[-6000:] + build.stderr[-6000:]
HOST_JAR = TREE / "bridge" / "host" / "build" / "libs" / "golem-host.jar"
MOD_JAR = TREE / "bridge" / "mod" / "build" / "libs" / ("golem-bridge-%s.jar" % WORKSPACE["version"])
assert HOST_JAR.is_file() and MOD_JAR.is_file(), (HOST_JAR, MOD_JAR)

## The game through the JNI without Minecraft

`golem run` starts any main class that the argument file of a server names, so the cell builds a
server without Minecraft: a main class that implements the interface `Game` of the host jar,
connects itself as the game, plays the server thread and runs `/golem` through the real natives
of `golem-bridge`. Each case of the notebook is an argument of the main class. The case `wait`
holds the server thread inside a question that waits for the one thread of a pool named
`Worker-Main-1`, while a chat message on that thread waits for the handler, as the load of a
chunk waits for the workers of the server. A call of another thread waits at most one second
for the handler, so the chat message must answer its pass value after that second, the task
behind it must run and the command must end, where a wait without end would hang the server. In
the case `throw` the question `location` throws a `StackOverflowError` three times, as a listener
of a mod that recurses would: each `/golem player` must answer that the game gives no answer, no
throwable may reach the main class, the log must name the error once and the stack trace must be
printed once, for the first failure alone. In the case `unread` the answer of `players` holds a
line with a game mode the game does not have beside a good one: the natives must drop that line
alone, so `/golem player` still finds the good player and not the broken one, and log the answer
at the first and the tenth of eleven questions. In the case `many` 2000 players are online, and
the natives must read them all, which needs a local frame that the question cleans as it reads.

In [ ]:
FAKE_JAVA = r"""
import com.skytekx.golem.host.Game;
import com.skytekx.golem.host.Host;
import java.util.concurrent.ExecutorService;
import java.util.concurrent.Executors;
import java.util.concurrent.Future;
import java.util.concurrent.TimeUnit;

/** A server without Minecraft whose game answers what each case of the notebook asks. */
public final class FakeServer implements Game {
	static final String US = "\u001F";
	static final String TESTER = "00000000-0000-4000-8000-000000000001";

	/** The pool that stands in for the workers of the server, with one thread. */
	final ExecutorService worker = Executors.newFixedThreadPool(1,
			task -> new Thread(task, "Worker-Main-1"));
	volatile String mode = "";

	static void say(String line) {
		System.out.println("fake: " + line);
		System.out.flush();
	}

	static long since(long start) {
		return (System.nanoTime() - start) / 1000000L;
	}

	static void show(String title, String[] lines) {
		say(title + " -> " + (lines == null ? "null" : lines.length + " lines"));
		if (lines != null) {
			for (String line : lines) {
				say("  " + line.replace(US, "|"));
			}
		}
	}

	static String player(String uuid, String name, String level) {
		return String.join(US, uuid, name, name, "0", "survival", level, "0", "");
	}

	static String uuid(int number) {
		return String.format("00000000-0000-4000-8000-%012d", number);
	}

	public String[] players() {
		if (mode.equals("many")) {
			String[] many = new String[2000];
			for (int index = 0; index < many.length; index++) {
				many[index] = player(uuid(index + 2), "P" + index, "0");
			}
			return many;
		}
		if (mode.equals("unread")) {
			return new String[] {player(TESTER, "Tester", "4"), String.join(US, uuid(2), "Broken",
					"Broken", "0", "hardcore", "0", "0", "")};
		}
		return new String[] {player(TESTER, "Tester", "4")};
	}

	public String location(String uuid) {
		if (mode.equals("throw")) {
			throw new StackOverflowError("fake: a listener recursed");
		}
		return String.join(US, "minecraft:overworld", "0.5", "64.0", "0.5", "0.0", "0.0");
	}

	public String respawn(String uuid) {
		return "";
	}

	public String spawn(String dimension) {
		return String.join(US, "0", "64", "0", "0.0");
	}

	public String mainHand(String uuid) {
		return "";
	}

	public String heights(String dimension) {
		return String.join(US, "-64", "320");
	}

	public String safeLocation(String dimension, double x, double y, double z, int radius) {
		if (mode.equals("wait")) {
			waitForWorker();
		}
		return String.join(US, "0.5", "64.0", "0.5");
	}

	/**
	 * Takes the one worker with a chat message, which waits for the handler that the server
	 * thread holds, then waits for a task behind it, as the load of a chunk waits for its task.
	 */
	void waitForWorker() {
		long start = System.nanoTime();
		worker.submit(() -> {
			boolean cancel = Host.chat(TESTER, "Tester", "hello");
			say("chat answered " + cancel + " after " + since(start) + " ms");
		});
		try {
			Thread.sleep(300);
			Future<String> task = worker.submit(() -> "ran");
			say("the task behind the chat " + task.get(30, TimeUnit.SECONDS) + " after "
					+ since(start) + " ms");
		} catch (Exception fault) {
			say("the task behind the chat never ran: " + fault);
		}
	}

	void waits() {
		mode = "wait";
		long start = System.nanoTime();
		show("teleport", Host.command("golem", null, null, 4,
				"teleport Tester minecraft:overworld 0 64 0"));
		say("the command took " + since(start) + " ms");
		mode = "";
	}

	void throwing() {
		mode = "throw";
		for (int round = 0; round < 3; round++) {
			show("player " + round, Host.command("golem", null, null, 4, "player Tester"));
		}
		mode = "";
	}

	void unread() {
		mode = "unread";
		for (int round = 0; round < 11; round++) {
			show("player " + round, Host.command("golem", null, null, 4, "player Tester"));
		}
		show("broken", Host.command("golem", null, null, 4, "player Broken"));
		mode = "";
	}

	void many() {
		mode = "many";
		show("many", Host.command("golem", null, null, 4, "player P1999"));
		mode = "";
	}

	public static void main(String[] cases) throws Exception {
		FakeServer game = new FakeServer();
		say("available " + Host.available());
		Host.connect(game);
		Host.serverStarting("47.4.18", "1.20.1");
		for (String name : cases) {
			say("case " + name);
			switch (name) {
			case "wait" -> game.waits();
			case "throw" -> game.throwing();
			case "unread" -> game.unread();
			case "many" -> game.many();
			default -> say("no case " + name);
			}
		}
		game.worker.shutdown();
		game.worker.awaitTermination(30, TimeUnit.SECONDS);
		Host.drain();
		Host.serverStopping();
		Host.serverStopped();
		say("done");
	}
}
"""

FAKE = SCRATCH / "fake"
(FAKE / "classes").mkdir(parents=True)
(FAKE / "FakeServer.java").write_text(FAKE_JAVA)
compiled = subprocess.run([str(JAVA_HOME / "bin" / "javac"), "--release", "17", "-cp", str(HOST_JAR), "-d",
                           str(FAKE / "classes"), str(FAKE / "FakeServer.java")], capture_output=True, text=True,
                          env=ENVIRONMENT)
assert compiled.returncode == 0, compiled.stdout + compiled.stderr
ARGS_DIR = FAKE / "libraries" / "net" / "minecraftforge" / "forge" / "0"
ARGS_DIR.mkdir(parents=True)
(ARGS_DIR / "unix_args.txt").write_text("-Xcheck:jni\n--add-modules ALL-MODULE-PATH\n-cp classes\nFakeServer\n")


def fake(*cases):
    # Runs the fake server under golem run with the cases, and answers its lines in their order.
    command = [GOLEM, "run", "--server", FAKE, "--java-home", JAVA_HOME, "--host-jar", HOST_JAR, "--", *cases]
    started = time.monotonic()
    ran = subprocess.run([str(part) for part in command], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env=ENVIRONMENT, timeout=120, stdin=subprocess.DEVNULL)
    lines = ran.stdout.splitlines()
    print("golem run %s: exit %d after %.1f s" % (" ".join(cases), ran.returncode, time.monotonic() - started))
    for line in lines:
        if line.startswith(("fake: ", "[golem] ")) or "Exception" in line or "Error" in line:
            print("  " + shown(line))
    assert ran.returncode == 0, shown(ran.stdout)
    assert "fake: done" in lines and lines[-1] == "[golem] the server ended: main returned and DestroyJavaVM answered 0"
    assert not [line for line in lines if " in native method" in line], shown(ran.stdout)
    return lines


def fake_numbers(lines, pattern):
    # The numbers of the lines of the fake server that match pattern, in their order.
    return [int(found.group(1)) for found in (re.search(pattern, line) for line in lines) if found]


lines = fake("wait")
chat_ms = fake_numbers(lines, r"^fake: chat answered false after (\d+) ms$")
task_ms = fake_numbers(lines, r"^fake: the task behind the chat ran after (\d+) ms$")
command_ms = fake_numbers(lines, r"^fake: the command took (\d+) ms$")
print("chat %s ms, task %s ms, command %s ms" % (chat_ms, task_ms, command_ms))
assert len(chat_ms) == 1 and 950 <= chat_ms[0] < 3000, chat_ms
assert len(task_ms) == 1 and chat_ms[0] <= task_ms[0] < 3000, task_ms
assert len(command_ms) == 1 and command_ms[0] < 3500, command_ms
assert "fake:   Queued a teleport of Tester to minecraft:overworld 0.50 64.00 0.50, yaw 0.0, pitch 0.0." in lines, lines
assert ("[golem] the native chat waited 1000 ms for the handler, which another call holds, so it answers the pass "
        "value, 1 times so far") in lines, lines

lines = fake("throw")
assert lines.count("fake:   The game gives no answer now, so try again.") == 3, lines
assert [line for line in lines if "the question location" in line] == [
    "[golem] the question location of the game threw java.lang.StackOverflowError: fake: a listener recursed, so it "
    "answers none, 1 times so far"], lines
assert lines.count("java.lang.StackOverflowError: fake: a listener recursed") == 1, "the trace is not printed once"
assert not [line for line in lines if line.startswith("Exception in thread")], "a throwable reached the main class"

lines = fake("unread", "many")
TESTER_LINE = "fake:   Tester (00000000-0000-4000-8000-000000000001): minecraft:overworld 0.50 64.00 0.50, yaw 0.0, pitch 0.0"
assert lines.count(TESTER_LINE) == 11, lines
assert "fake:   No player Broken is online." in lines, lines
broken = ("[golem] the question players of the game answered \"00000000-0000-4000-8000-000000000002\\u{1f}Broken\\u{1f}"
          "Broken\\u{1f}0\\u{1f}hardcore\\u{1f}0\\u{1f}0\\u{1f}\", which does not read, so it answers none, %d times so far")
assert [line for line in lines if "the question players" in line] == [broken % 1, broken % 10], lines
assert ("fake:   P1999 (00000000-0000-4000-8000-000000002001): minecraft:overworld 0.50 64.00 0.50, yaw 0.0, "
        "pitch 0.0") in lines, lines

## A copy of the server under golem

The cell copies the installed server without its world, logs, crash reports and mods, sets the
port of the copy, turns command blocks on and names the permission handler of Golem,
`golem:permissions`, in `defaultconfigs/forge-server.toml`, which Forge copies into the
configuration of the new world. A boot runs `golem run` on the copy with the Java home, the host
jar and `nogui` after `--`, reads its output on a thread, waits for `Done` and answers what it
printed and logged. Every line of the host starts with `[golem]` on standard error, which the boot
reads together with the console of the server.

In [ ]:
TOP = {"world", "logs", "crash-reports", "mods"}


def copy_server(name, with_mod):
    server = SCRATCH / name
    shutil.copytree(FORGE, server, symlinks=True,
                    ignore=lambda directory, names: [entry for entry in names if Path(directory) == FORGE
                                                     and (entry in TOP or entry.endswith(".log"))])
    (server / "mods").mkdir()
    if with_mod:
        shutil.copy2(MOD_JAR, server / "mods" / MOD_JAR.name)
    settings = [line for line in (server / "server.properties").read_text().splitlines()
                if not line.startswith(("server-port=", "server-ip=", "online-mode=", "enable-command-block="))]
    settings += ["server-port=%d" % server_port, "server-ip=127.0.0.1", "online-mode=false",
                 "enable-command-block=true"]
    (server / "server.properties").write_text("\n".join(settings) + "\n")
    (server / "defaultconfigs").mkdir(exist_ok=True)
    (server / "defaultconfigs" / "forge-server.toml").write_text('[server]\n\tpermissionHandler = "golem:permissions"\n')
    return server


HANDLER = "Successfully initialized permission handler golem:permissions"
NO_HANDLER = "Unable to find configured permission handler golem:permissions, will use forge:default_handler"


class Boot:
    # One run of golem on a copy of the server, with its output read on a thread as it comes.

    def __init__(self, server, ignored=()):
        self.server = server
        command = [GOLEM, "run", "--server", server, "--java-home", JAVA_HOME, "--host-jar", HOST_JAR, "--", "nogui"]
        self.lines = []
        # The signals of `ignored` start ignored, as nohup leaves SIGHUP.
        ignore = (lambda: [signal.signal(number, signal.SIG_IGN) for number in ignored]) if ignored else None
        self.process = subprocess.Popen([str(part) for part in command], cwd=SCRATCH, stdin=subprocess.PIPE,
                                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=ENVIRONMENT,
                                        preexec_fn=ignore)
        self.reader = threading.Thread(target=self.pump, daemon=True)
        self.reader.start()
        self.started = time.monotonic()
        self.expect("]: Done (", timeout=boot_timeout)
        self.done = time.monotonic() - self.started

    def pump(self):
        for line in self.process.stdout:
            self.lines.append(line.rstrip("\n"))

    def tail(self, count=60):
        return shown("\n".join(self.lines[-count:]))

    def expect(self, text, count=1, timeout=60):
        # Waits until `count` lines of the output hold `text`, and answers the last of them.
        deadline = time.monotonic() + timeout
        while True:
            found = [line for line in list(self.lines) if text in line]
            if len(found) >= count:
                return found[-1]
            if self.process.poll() is not None:
                raise AssertionError("golem ended with %s before %r:\n%s" % (self.process.returncode, text, self.tail()))
            if time.monotonic() > deadline:
                self.process.kill()
                raise AssertionError("no %r after %d s:\n%s" % (text, timeout, self.tail()))
            time.sleep(0.2)

    def console(self, line):
        self.process.stdin.write(line + "\n")
        self.process.stdin.flush()

    def signal(self, number):
        self.process.send_signal(number)

    def wait(self):
        try:
            self.code = self.process.wait(boot_timeout)
        except subprocess.TimeoutExpired:
            self.process.kill()
            raise AssertionError("golem did not end:\n" + self.tail())
        self.reader.join(10)
        self.latest = (self.server / "logs" / "latest.log").read_text(errors="replace")
        crashes = self.server / "crash-reports"
        self.crashes = sorted(path.name for path in crashes.iterdir()) if crashes.is_dir() else []
        self.fatal = [line for line in self.latest.splitlines() if "/FATAL]" in line]
        print("exit %s, Done after %.1f s, FATAL lines %d, crash reports %d" % (
            self.code, self.done, len(self.fatal), len(self.crashes)))
        return self

    def host(self):
        # The lines of the host, without their prefix.
        return [line[len("[golem] "):] for line in self.lines if line.startswith("[golem] ")]


def until(test, timeout=30):
    # Waits until test answers true, and answers whether it did.
    deadline = time.monotonic() + timeout
    while not test():
        if time.monotonic() > deadline:
            return False
        time.sleep(0.1)
    return True


print("boot ready")

## The bridge connects

The server with the mod must reach `Done`, and the core plugin must log once that the bridge is
connected with the versions of Forge and Minecraft, and Forge must take the permission handler of
Golem, so the mod warns of no other handler. The host names its version, its commit and the main
class of Forge before the JVM runs. On the console, `golem source` must name the license, the
repository and the tree of the commit of the binary, which section 13 of the AGPL asks for, and
`golem status` the ticks that the bridge counted.

In [ ]:
WITH_MOD = copy_server("with-mod", with_mod=True)
on = Boot(WITH_MOD)
on.expect("[golem] the bridge is connected: Forge %s, Minecraft %s" % (forge_version, minecraft_version))
on.expect(HANDLER)
assert not [line for line in on.lines if "Forge answers the PermissionAPI with" in line], on.tail()
first = on.host()[0]
print(shown(first))
assert first.startswith("Golem %s at %s runs cpw.mods.bootstraplauncher.BootstrapLauncher of " % (
    WORKSPACE["version"], COMMIT)), first
tree = "%s/tree/%s" % (WORKSPACE["repository"], COMMIT) if COMMIT != "unknown" else WORKSPACE["repository"]
on.console("golem source")
on.expect("Golem %s is free software under the GNU Affero General Public License 3.0 only, AGPL-3.0-only." % WORKSPACE["version"])
on.expect("Repository: " + WORKSPACE["repository"])
if COMMIT != "unknown":
    on.expect("Source of this build: " + tree)
on.console("golem status")
status = on.expect("Golem %s: TPS " % WORKSPACE["version"])
print(status.split("]: ", 1)[1])
connected = [line for line in on.host() if line.startswith("the bridge is connected")]
assert len(connected) == 1, connected

## A player

A player joins in offline mode, chats and runs `/golem source` and `/golem status`. The chat must
reach the log of the server, so the natives called on the worker thread of Forge let it pass, and
the player must receive the lines of the core plugin as system messages, with the link to the tree
of the commit. The console sets the game rule `sendCommandFeedback` to false first, which silences
the feedback of commands, so the lines must still arrive, since they are system messages. The
player is no operator, so the selector `@s` asks the permission handler of Golem for
`forge.use_entity_selectors`. No plugin decides that node, so it must fall to its default
resolver, which refuses a player below level 2.

In [ ]:
player = Client(server_port, "Tester")
player.login()
on.expect("Tester joined the game")
assert player.uuid == offline_uuid("Tester"), player.uuid
on.console("gamerule sendCommandFeedback false")
on.expect("Gamerule sendCommandFeedback is now set to: false")
player.chat("hello golem")
on.expect("<Tester> hello golem")
player.command("golem source")
assert player.heard("AGPL-3.0-only"), "the player never heard the license: %s" % player.messages
if COMMIT != "unknown":
    assert player.heard(tree), "the player never heard the tree of the commit: %s" % player.messages
player.command("golem status")
assert player.heard("TPS"), "the player never heard the status: %s" % player.messages
player.command("msg @s hello")
assert player.heard("argument.entity.selector.not_allowed"), player.messages
assert "hello" not in player.chats, player.chats
assert not any("Host.permission failed" in line for line in on.lines), on.tail()
print("system messages of the player: %d" % len(player.messages))

## Who runs a command

The natives must tell the sources of a command apart. `/golem whoami` must name the console for the
console, an armor stand at level 4 when the console runs it as the stand, and a command block at
level 2, whose last output `data get block` shows. A command that the console or a command block
runs as the player through `execute as` must reach Golem as that source, named after the player, at
the level of the source, 4 and 2, so no source borrows the rights of a player, and its answer goes
to the player. A player below level 2 must be refused the tools of operators.

In [ ]:
TESTER = offline_uuid("Tester")
on.console("golem whoami")
on.expect("You run /golem as the console.")
on.console("execute as Tester run golem whoami")
assert player.heard("You run /golem as Tester at the level 4."), player.messages
on.console("summon minecraft:armor_stand")
on.expect("Summoned new Armor Stand")
on.console("execute as @e[type=minecraft:armor_stand,limit=1] run golem whoami")
on.expect("You run /golem as Armor Stand at the level 4.")
on.console("forceload add 0 0")
on.expect("to be force loaded")
on.console('setblock 0 200 0 minecraft:command_block{Command:"golem whoami"}')
on.expect("Changed the block at 0, 200, 0")
on.console("setblock 0 201 0 minecraft:redstone_block")
on.expect("Changed the block at 0, 201, 0")


def last_output():
    on.console("data get block 0 200 0 LastOutput")
    time.sleep(0.5)
    return any("You run /golem as @ at the level 2." in line for line in on.lines)


assert until(last_output), on.tail()
on.console('setblock 2 200 0 minecraft:command_block{Command:"execute as Tester run golem whoami"}')
on.expect("Changed the block at 2, 200, 0")
on.console("setblock 2 201 0 minecraft:redstone_block")
on.expect("Changed the block at 2, 201, 0")
assert player.heard("You run /golem as Tester at the level 2."), player.messages
player.command("golem player Tester")
assert player.heard("Only the console and operators of level 2 and more may run /golem player."), player.messages
print("the console, the player, the armor stand and the command block each ran as themselves")

## The game

After `op Tester` the player is an operator of level 4. `/golem player` must read the game through
the real natives: the place that `data get entity` shows, the game mode, the operator level, the
respawn point and the world spawn that the console sets and the diamonds that it gives.
`/golem world` must name the world spawn for the Overworld and the Nether, which shares it, with the
heights of each. The respawn point of `/spawnpoint` is no bed, and it is the point that the player
holds as Minecraft stores it: once the console fills its block with obsidian, `/golem player` must
still name it, since Minecraft checks the block only on a respawn, and after the player died and
respawned, which finds the block blocked and clears the point, it must name none.

In [ ]:
on.console("op Tester")
on.expect("Made Tester a server operator")
for command in ("spawnpoint Tester 10 64 10", "give Tester minecraft:diamond 3", "setworldspawn 4 70 4"):
    on.console(command)
on.expect("Set the world spawn point to 4, 70, 4")
on.console("data get entity Tester Pos")
pos = on.expect("Tester has the following entity data: [")
x, y, z = (float(number.rstrip("d")) for number in pos.rsplit("[", 1)[1].rstrip("]").split(", "))
player.command("golem player Tester")
where = "Tester (%s): minecraft:overworld %.2f %.2f %.2f, yaw 0.0, pitch 0.0" % (TESTER, x, y, z)
for line in (where, "Game mode survival, ping ", ", operator level 4, idle ", "Display name: Tester",
             "List name: none", "Respawn point: minecraft:overworld 10 64 10", "Main hand: 3 minecraft:diamond"):
    assert player.heard(line), "the player never heard %r: %s" % (line, player.messages)
on.console("fill 10 64 10 10 65 10 minecraft:obsidian")
on.expect("Successfully filled 2 block")
on.console("golem player Tester")
on.expect("Respawn point: minecraft:overworld 10 64 10")
respawns = len(player.dimensions)
on.console("kill Tester")
on.expect("Killed Tester")
player.respawn()
assert until(lambda: len(player.dimensions) > respawns), "the player never respawned"
on.console("golem player Tester")
on.expect("Respawn point: none")
on.console("golem world")
on.expect("minecraft:overworld: spawn 4 70 4 at the angle 0.0, blocks from y -64 below y 320")
on.console("golem world minecraft:the_nether")
on.expect("minecraft:the_nether: spawn 4 70 4 at the angle 0.0, blocks from y 0 below y 256")
print(where)

## A teleport to a safe place

The console builds a floor of obsidian in the Nether with a block of magma in its middle, and a
solid box of obsidian beside it. A teleport onto the magma must land on the nearest safe block, the
obsidian at the lower x, with the rotation the player has, in the Nether, as the game and the
client both show. A teleport into the box must find no safe place within 3 blocks and leave the
player where they are, and a teleport into a dimension the server does not have must say so. A
teleport to a chunk of the Overworld that was never generated must answer that its chunks are not
loaded, since a question loads no chunk, and the world must hold no region of it after the stop.

In [ ]:
NETHER = "execute in minecraft:the_nether run "
for command in ("forceload add -3 -3 20 20", "fill -3 99 -3 3 104 3 minecraft:air",
                "fill -3 99 -3 3 99 3 minecraft:obsidian", "setblock 0 99 0 minecraft:magma_block",
                "fill 10 99 10 20 109 20 minecraft:obsidian"):
    on.console(NETHER + command)
on.console("say the fixture is ready")
on.expect("[Server] the fixture is ready")
player.command("golem teleport Tester minecraft:the_nether 0.5 100 0.5")
landed = "Queued a teleport of Tester to minecraft:the_nether -0.50 100.00 0.50, yaw 0.0, pitch 0.0."
assert player.heard(landed), player.messages
assert until(lambda: player.dimensions[-1:] == ["minecraft:the_nether"]), player.dimensions
assert until(lambda: (-0.5, 100.0, 0.5, 0.0, 0.0) in player.positions), player.positions
player.command("golem player Tester")
assert player.heard("Tester (%s): minecraft:the_nether -0.50 100.00 0.50" % TESTER), player.messages
player.command("golem teleport Tester minecraft:the_nether 15.5 104 15.5")
assert player.heard("No safe place within 3 blocks of minecraft:the_nether 15.50 104.00 15.50, yaw 0.0, "
                    "pitch 0.0, so Tester stays."), player.messages
on.console("golem teleport Tester golem:void 0 64 0")
on.expect("The server has no dimension golem:void, or the game gives no answer now.")
FAR = 100000
player.command("golem teleport Tester minecraft:overworld %d.5 64 %d.5" % (FAR, FAR))
assert player.heard("The chunks near minecraft:overworld %d.50 64.00 %d.50, yaw 0.0, pitch 0.0 are not loaded, so "
                    "Tester stays, since the search for a safe place loads no chunk." % (FAR, FAR)), player.messages
on.console("execute in minecraft:the_nether run data get entity Tester Pos")
stayed = on.expect("Tester has the following entity data: [", count=2)
assert stayed.endswith("[-0.5d, 100.0d, 0.5d]"), stayed
print(landed)

## The rules of the safe place

Above the roof of the Nether, in the chunks that the console keeps loaded, the console builds one
fixture per rule, each in a box of air of its own, so no search reaches the next one, and the
teleport of `/golem teleport` must land where the rules of `okf/bridge.md` say. A nether portal in
a frame, an end portal and an end gateway on a floor of obsidian and a cobweb let a player through
and hurt no one, but a portal carries the player away and a cobweb holds them, so a target in each
must land beside it. The order of the search, the nearest place first, then the higher, then the
lower x, then the lower z, must decide between places at the same distance: obsidian at the feet
of the target with the floor beside it must land on the obsidian, one block higher, and not on
the floor at the lower x, magma at the feet must land at the lower x and not at the lower z, and
with the lower x walled must land at the lower z. Lava and water at the feet, held in by walls,
must land on a wall, fire on netherrack must land beside it, a target on obsidian at y 254 of the
Nether, whose head would reach the top of the world, must find no place, and a target at y 0,
the lowest block of the Nether, in a shell of obsidian must land one block higher. Under a roof
of bedrock that the console lays at y 127 with air below and above it, a target at y 126 lies
below the logical height of the Nether, y 128, so the search must not land on the roof, while a
target at y 128 above the same roof must. Last, the console sets the world border to 20 blocks
around `worldborder center 0 0`, which lies on the middle of the block 0 0, so the border runs
through the middle of the blocks at x 10 and -10. A target on the block at x 10 must land on the
block at x 9, where a player stands wholly inside the border, and the border is set back.

In [ ]:
def build(*commands):
    # Builds a fixture in the Nether with the console and waits until the server ran it.
    build.count += 1
    for command in commands:
        on.console(NETHER + command)
    on.console("say fixture %d is ready" % build.count)
    on.expect("[Server] fixture %d is ready" % build.count)


build.count = 0


def box(x, z, y=146):
    # Clears a box of air around the column x, z and lays a floor of obsidian at y 149 in it.
    return ["fill %d %d %d %d %d %d minecraft:air" % (x - 4, y, z - 4, x + 4, y + 10, z + 4),
            "fill %d 149 %d %d 149 %d minecraft:obsidian" % (x - 3, z - 3, x + 3, z + 3)]


def teleport(x, y, z, dimension="minecraft:the_nether"):
    player.command("golem teleport Tester %s %s %s %s" % (dimension, x, y, z))


def landed(x, y, z, dimension="minecraft:the_nether"):
    return "Queued a teleport of Tester to %s %.2f %.2f %.2f, yaw 0.0, pitch 0.0." % (dimension, x, y, z)


build(*box(-12, -12), "fill -13 150 -12 -13 153 -12 minecraft:obsidian",
      "fill -10 150 -12 -10 153 -12 minecraft:obsidian", "fill -12 153 -12 -11 153 -12 minecraft:obsidian",
      "fill -12 150 -12 -11 152 -12 minecraft:nether_portal[axis=x]",
      *box(-4, -12), "setblock -4 150 -12 minecraft:end_portal",
      *box(4, -12), "setblock 4 150 -12 minecraft:cobweb",
      *box(12, -12), "setblock 12 150 -12 minecraft:end_gateway")
cases = [((-11.5, 150, -11.5), (-11.5, 150.0, -12.5), "a nether portal: beside the frame, outside the portal"),
         ((-3.5, 150, -11.5), (-4.5, 150.0, -11.5), "an end portal: on the obsidian beside it"),
         ((4.5, 150, -11.5), (3.5, 150.0, -11.5), "a cobweb: on the obsidian beside it"),
         ((12.5, 150, -11.5), (11.5, 150.0, -11.5), "an end gateway: on the obsidian beside it")]
for target, place, what in cases:
    teleport(*target)
    assert player.heard(landed(*place)), "%s: %s" % (what, player.messages[-3:])
    print("%-55s %s" % (what, landed(*place)))
assert player.dimensions[-1] == "minecraft:the_nether", player.dimensions

build(*box(-12, 12), "setblock -12 150 12 minecraft:obsidian",
      *box(-4, 12), "setblock -4 150 12 minecraft:magma_block",
      *box(4, 12), "setblock 4 150 12 minecraft:magma_block", "setblock 3 150 12 minecraft:obsidian",
      *box(12, 12), "fill 11 150 12 13 150 12 minecraft:obsidian", "fill 12 150 11 12 150 13 minecraft:obsidian",
      "setblock 12 150 12 minecraft:lava",
      *box(20, 12), "fill 19 150 12 21 150 12 minecraft:obsidian", "fill 20 150 11 20 150 13 minecraft:obsidian",
      "setblock 20 150 12 minecraft:water",
      *box(-12, 20), "setblock -12 149 20 minecraft:netherrack", "setblock -12 150 20 minecraft:fire",
      "setblock -4 254 20 minecraft:obsidian",
      "fill -1 0 16 9 6 24 minecraft:obsidian hollow")
cases = [((-11.5, 150, 12.5), (-11.5, 151.0, 12.5), "the higher before the lower x"),
         ((-3.5, 150, 12.5), (-4.5, 150.0, 12.5), "the lower x before the lower z"),
         ((4.5, 150, 12.5), (4.5, 150.0, 11.5), "the lower z before the higher z and x"),
         ((12.5, 150, 12.5), (11.5, 151.0, 12.5), "lava at the feet: on the wall at the lower x"),
         ((20.5, 150, 12.5), (19.5, 151.0, 12.5), "water at the feet: on the wall at the lower x"),
         ((-11.5, 150, 20.5), (-12.5, 150.0, 20.5), "fire at the feet: beside it"),
         ((4.5, 0, 20.5), (4.5, 1.0, 20.5), "the lowest block of the Nether: one block higher")]
for target, place, what in cases:
    teleport(*target)
    assert player.heard(landed(*place)), "%s: %s" % (what, player.messages[-3:])
    print("%-55s %s" % (what, landed(*place)))


def stays(x, y, z, dimension="minecraft:the_nether"):
    return "No safe place within 3 blocks of %s %.2f %.2f %.2f, yaw 0.0, pitch 0.0, so Tester stays." % (
        dimension, x, y, z)


teleport(-3.5, 255, 20.5)
assert player.heard(stays(-3.5, 255, 20.5)), "the head reached the top of the world: %s" % player.messages[-3:]
print("%-55s %s" % ("the top of the world", stays(-3.5, 255, 20.5)))
build("fill -16 118 0 -8 134 8 minecraft:air", "fill -15 127 1 -9 127 7 minecraft:bedrock")
teleport(-11.5, 126, 4.5)
assert player.heard(stays(-11.5, 126, 4.5)), "the search landed on the roof: %s" % player.messages[-3:]
print("%-55s %s" % ("under the roof of the Nether", stays(-11.5, 126, 4.5)))
teleport(-11.5, 128, 4.5)
assert player.heard(landed(-11.5, 128.0, 4.5)), player.messages[-3:]
print("%-55s %s" % ("on the roof of the Nether", landed(-11.5, 128.0, 4.5)))

for command in ("fill 5 196 -4 15 206 4 minecraft:air", "fill 6 199 -3 14 199 3 minecraft:obsidian",
                "worldborder center 0 0", "worldborder set 20"):
    on.console(command)
on.expect("Set the center of the world border to 0.50, 0.50")
on.expect("Set the world border to 20.0 block(s) wide")
teleport(10.5, 200, 0.5, "minecraft:overworld")
assert player.heard(landed(9.5, 200.0, 0.5, "minecraft:overworld")), "half the player stands outside the border: %s" % (
    player.messages[-3:],)
print("%-55s %s" % ("the world border through the middle of a block", landed(9.5, 200.0, 0.5, "minecraft:overworld")))
on.console("worldborder set 59999968")
on.expect("Set the world border to 59999968.0 block(s) wide")

## The name in the player list

`/golem listname` must set the name of the player in the player list, which the client sees and
`/golem player` names, and clear it again. No question of the game may have failed or answered
what the natives cannot read on the way.

In [ ]:
player.command("golem listname Tester [AFK] Tester")
assert player.heard("The player list names Tester [AFK] Tester."), player.messages
assert until(lambda: "[AFK] Tester" in (player.list_names.get(TESTER) or "")), player.list_names
player.command("golem player Tester")
assert player.heard("List name: [AFK] Tester"), player.messages
player.command("golem listname Tester")
assert until(lambda: player.list_names.get(TESTER, "") is None), player.list_names
assert not any(line.startswith("the question ") for line in on.host()), on.host()
print("the list name was set and cleared, the client saw both")

## The stop by a signal

A `SIGQUIT` first must print the dump of every thread on the console and stop nothing, as it does
under java, although `-Xrs` takes the signal from the JVM. Then `SIGTERM` must queue
`save-all flush` and `stop` for the server thread, and a `SIGINT` behind it must only log that the
stop is queued. The mod runs both commands as the console at the end of the
next tick, so the game is saved before the server stops, and the player still online is
disconnected. `golem run` must exit with 0 after `DestroyJavaVM` answered 0, with no fatal line and
no crash report. The search for a safe place far away must have left no region of the world
behind.

In [ ]:
on.signal(signal.SIGQUIT)
on.expect("[golem] SIGQUIT: printed the dump of every thread", timeout=30)
dump = [line for line in on.lines if line.startswith("Full thread dump OpenJDK")]
print(dump[0] if dump else "no dump")
assert dump, "SIGQUIT printed no dump of the threads"
assert any(line.startswith('"Server thread"') for line in on.lines), "the dump lacks the server thread"
time.sleep(2)
assert on.process.poll() is None, "SIGQUIT ended golem"
on.signal(signal.SIGTERM)
on.expect("[golem] SIGTERM: queued save-all flush and stop on the server thread", timeout=10)
on.signal(signal.SIGINT)
on.expect("[golem] SIGINT: the stop is already queued, waiting for the server to end", timeout=10)
on.wait()
assert player.closed.wait(30), "the player was never disconnected"
player.close()
print(shown("\n".join(on.host()[-4:])))
assert on.code == 0, "golem exited %s:\n%s" % (on.code, on.tail())
assert on.host()[-1] == "the server ended: main returned and DestroyJavaVM answered 0", on.host()
assert not on.fatal and not on.crashes, (on.fatal, on.crashes)
order = [text for text in ("Saving the game", "Stopping the server", "Saving worlds", "All dimensions are saved")
         if text in on.latest]
assert order == ["Saving the game", "Stopping the server", "Saving worlds", "All dimensions are saved"], order
assert on.latest.index("Saving the game") < on.latest.index("Stopping the server"), "save-all ran after stop"
assert (WITH_MOD / "world" / "level.dat").is_file(), "the world was not saved"
far_region = WITH_MOD / "world" / "region" / ("r.%d.%d.mca" % (FAR >> 9, FAR >> 9))
assert not far_region.exists(), "the search for a safe place generated %s" % far_region.name

## A server without the bridge

The second copy has no mod in `mods/`, so the natives are registered but nothing calls them and no
tick drains the queue. Golem starts with `SIGHUP` ignored, as under `nohup`, so a hangup must not
stop the server, as it does not stop a JVM under nohup. The server must still reach `Done` as a
stock Forge server, with the default permission handler of Forge in place of `golem:permissions`,
and `SIGTERM` must end the JVM through `System.exit(143)`: the shutdown hook of the server stops it
and saves the worlds, and the exit hook of the host ends golem with 143, as the java launcher
would.

In [ ]:
WITHOUT_MOD = copy_server("without-mod", with_mod=False)
off = Boot(WITHOUT_MOD, ignored=[signal.SIGHUP])
off.expect("[golem] SIGHUP is ignored, as the process started with it ignored, so it does not stop the server")
off.expect(NO_HANDLER)
off.signal(signal.SIGHUP)
time.sleep(3)
assert off.process.poll() is None, "SIGHUP ended golem under nohup"
assert not any(line.startswith("[golem] SIGHUP:") for line in off.lines), off.host()
off.signal(signal.SIGTERM)
off.expect("[golem] SIGTERM: the bridge is not connected, so the JVM exits through the shutdown hook of the server",
           timeout=10)
off.wait()
print(shown("\n".join(off.host())))
assert not any(line.startswith("the bridge is connected") for line in off.host()), off.host()
assert off.code == 128 + signal.SIGTERM, "golem exited %s:\n%s" % (off.code, off.tail())
assert off.host()[-1] == "the JVM exits with status 143", off.host()
assert not off.fatal and not off.crashes, (off.fatal, off.crashes)
for text in ("Stopping server", "Saving worlds", "All dimensions are saved"):
    assert text in off.latest, "the log lacks %r" % text
assert (WITHOUT_MOD / "world" / "level.dat").is_file(), "the world was not saved"

## Verdict

The cell clears the temporary directory. A run that reaches it passed.

In [ ]:
shutil.rmtree(SCRATCH)
print("verdict: golem run boots Forge %s with the bridge connected in %.0f s, answers golem source and status on the "
      "console and to a player, tells the console, a player, an armor stand and a command block apart, reads the game "
      "and teleports and names a player through the tools of operators, dumps its threads on SIGQUIT, stops on SIGTERM through the server thread with exit 0, and stops a server without "
      "the bridge through its shutdown hook with exit 143, which a hangup under nohup does not stop" % (forge_version, on.done))